# Solver Comparison and Smoke Tests

Legacy solver comparison recipes, smoke-test harnesses, and runtime summary analysis.

All cells are intentionally unexecuted; run sections selectively from top to bottom.


In [ ]:
from pathlib import Path
import os

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = Path.cwd().parent
os.chdir(ROOT)
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
ARTIFACTS = ROOT / 'artifacts'
ARTIFACTS.mkdir(exist_ok=True)


## `exps/solve.py`

Single model/solver runtime recipe.


In [ ]:
import argparse
import sys
import time
from pathlib import Path

import numpy as np


from mdpforge.utils.data_management import import_models_from_file, import_solver_from_file


def matrix_nnz(matrix):
    return matrix.nnz if hasattr(matrix, "nnz") else np.count_nonzero(matrix)


def transition_density(model):
    nnz = sum(matrix_nnz(matrix) for matrix in model.transition_matrix)
    return nnz / model.action_dim / model.state_dim**2


def print_model_summary(model, build_time):
    transition_nnz = sum(matrix_nnz(matrix) for matrix in model.transition_matrix)

    print("Model")
    print(f"  name: {model.name}")
    print(f"  states: {model.state_dim}")
    print(f"  actions: {model.action_dim}")
    print(f"  transition nonzeros: {transition_nnz}")
    print(f"  transition density: {transition_density(model):.6g}")
    print(f"  build time: {build_time:.6f}s")


def print_solver_summary(args, solver, runtimes):
    print("Solver")
    print(f"  name: {args.solver}")
    print(f"  discount: {args.discount}")
    print(f"  precision: {args.precision}")
    print(f"  repeats: {args.repeat}")
    print(f"  mean runtime: {np.mean(runtimes):.6f}s")
    print(f"  std runtime: {np.std(runtimes):.6f}s")

    if hasattr(solver, "partition"):
        print(f"  regions: {solver.partition.n_regions}")


parser = argparse.ArgumentParser()
parser.add_argument("--model", type=str, required=True)
parser.add_argument("--solver", type=str, required=True)
parser.add_argument("--state", type=int, required=True)
parser.add_argument("--discount", type=float, required=True)
parser.add_argument("--action", type=int, default=10)
parser.add_argument("--precision", type=float, default=1e-2)
parser.add_argument("--repeat", type=int, default=1)
parser.add_argument("--check-transition", action="store_true")
parser.add_argument("--no-save", action="store_true")
args = parser.parse_args()

if args.repeat <= 0:
    raise ValueError("--repeat must be positive.")

model_class = import_models_from_file(args.model)
model = model_class(args.state, args.action)

start = time.perf_counter()
model.create_model(check_transition=args.check_transition, save=not args.no_save)
build_time = time.perf_counter() - start

solver = import_solver_from_file(args.solver)
solver.__init__(model, args.discount, args.precision)

runtimes = []
for repeat_index in range(args.repeat):
    start = time.perf_counter()
    solver.run()
    elapsed = time.perf_counter() - start
    runtime = getattr(solver, "runtime", elapsed)
    runtime = elapsed if runtime is None else runtime
    runtimes.append(runtime)

print_model_summary(model, build_time)
print()
print_solver_summary(args, solver, runtimes)


## `exps/comp.py`

Batch runtime comparison across models.


In [ ]:
import argparse
import csv
import importlib.util
import sys
import time
from pathlib import Path


ROOT = Path(__file__).resolve().parent

from mdpforge.utils.paths import MODEL_PATH


def load_module(path: Path):
    spec = importlib.util.spec_from_file_location(path.stem, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def create_model(model):
    try:
        model.create_model(save=False)
    except TypeError:
        model.create_model()


def solver_runtime(solver_class, model, discount, precision):
    start = time.perf_counter()
    solver = solver_class(model, discount, precision)
    solver.run()
    elapsed = time.perf_counter() - start
    return getattr(solver, "runtime", elapsed) or elapsed


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--state", type=int, default=100)
    parser.add_argument("--action", type=int, default=10)
    parser.add_argument("--discount", type=float, default=0.99)
    parser.add_argument("--precision", type=float, default=1e-2)
    parser.add_argument("--output", type=Path, default=ROOT / "comp_results.csv")
    parser.add_argument("--quiet", action="store_true")
    args = parser.parse_args()

    from mdpforge.solvers.aggregated_vi import Solver as AggregatedVI
    from mdpforge.solvers.aggregated_qvi import Solver as AggregatedQVI
    from mdpforge.solvers.aggregated_pim import Solver as AggregatedPIM
    from mdpforge.solvers.personal_vi import Solver as PersonalVI
    from mdpforge.solvers.personal_pim import Solver as PersonalPIM

    solvers = [
        ("aggregated_vi", AggregatedVI),
        ("aggregated_qvi", AggregatedQVI),
        ("aggregated_pim", AggregatedPIM),
        ("personal_vi", PersonalVI),
        ("personal_pim", PersonalPIM),
    ]
    header = ["model"] + [solver_name for solver_name, _ in solvers]

    model_files = sorted(
        path for path in MODEL_PATH.glob("*.py") if path.name != "__init__.py"
    )

    if not args.quiet:
        print(f"Writing results to {args.output}")
        print(",".join(header))

    with args.output.open("w", newline="") as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(header)

        for model_file in model_files:
            try:
                module = load_module(model_file)
                model = module.Model(args.state, args.action)
                create_model(model)

                runtimes = []
                for _, solver_class in solvers:
                    runtime = solver_runtime(
                        solver_class, model, args.discount, args.precision
                    )
                    runtimes.append(f"{runtime:.6f}")

                row = [model_file.stem] + runtimes
                writer.writerow(row)
                csv_file.flush()
                if not args.quiet:
                    print(",".join(row), flush=True)
            except Exception:
                row = [model_file.stem] + ["failed"] * len(solvers)
                writer.writerow(row)
                csv_file.flush()
                if not args.quiet:
                    print(",".join(row), flush=True)


if __name__ == "__main__":
    main()


## `exps/analyze.py`

Runtime CSV summary and winner analysis.


In [ ]:
import pandas as pd
import numpy as np

csv_path = "comp_results.csv"

df = pd.read_csv(csv_path)

solver_cols = [c for c in df.columns if c != "model"]

# Convert failed / non-numeric entries to NaN
df[solver_cols] = df[solver_cols].apply(pd.to_numeric, errors="coerce")

# Keep rows where at least one solver succeeded
valid = df.dropna(subset=solver_cols, how="all").copy()

# Runtime summaries
summary = pd.DataFrame({
    "mean": valid[solver_cols].mean(),
    "median": valid[solver_cols].median(),
    "geomean": valid[solver_cols].apply(
        lambda x: np.exp(np.log(x.dropna()).mean())
    ),
    "num_success": valid[solver_cols].count(),
}).sort_values("geomean")

print("\nRuntime summary:")
print(summary)

# Winner per benchmark instance
winners = valid[solver_cols].idxmin(axis=1)

winning_runtime = valid[solver_cols].to_numpy()[
    np.arange(len(valid)),
    valid[solver_cols].columns.get_indexer(winners),
]

winner_table = pd.DataFrame({
    "model": valid["model"].to_numpy(),
    "winner": winners.to_numpy(),
    "winning_runtime": winning_runtime,
})

print("\nWinner per model:")
print(winner_table.to_string(index=False))

# Win counts and percentages
win_stats = (
    winners.value_counts()
    .reindex(solver_cols, fill_value=0)
    .to_frame("num_first")
)

win_stats["pct_first"] = 100 * win_stats["num_first"] / len(valid)

win_stats = win_stats.sort_values(
    ["num_first", "pct_first"],
    ascending=False,
)

print("\nWin-rate statistics:")
print(win_stats)


## `exps/test_all_solvers_common.py`

Shared legacy solver-suite harness.


In [ ]:
import argparse
import importlib.util
import inspect
import os
import sys
import time
from pathlib import Path
from typing import Callable

import numpy as np

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

ROOT = Path(__file__).resolve().parents[1]

from mdpforge.utils.exact_value_function import get_exact_value
from mdpforge.utils.paths import SOLVER_PATH


OPTIONAL_BACKEND_IMPORT_ERRORS = (
    "DLL load failed",
    "No module named",
)


RUN_PARAMS = {
    "num_episodes": 10,
    "episode_length": 10,
    "num_replay_passes": 1,
    "log_every": 1000000,
    "refine_interval": 0,
}

def load_module(path: Path):
    spec = importlib.util.spec_from_file_location(path.stem.replace(" ", "_"), path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def is_optional_backend_import_error(exc: Exception) -> bool:
    return isinstance(exc, (ImportError, ModuleNotFoundError)) and any(
        message in str(exc) for message in OPTIONAL_BACKEND_IMPORT_ERRORS
    )


def build_solver(solver_class, model, discount: float, precision: float):
    signature = inspect.signature(solver_class)
    if "discount" in signature.parameters:
        return solver_class(model, discount, precision)
    return solver_class(model, precision)


def run_solver(solver):
    signature = inspect.signature(solver.run)
    if "run_params" in signature.parameters:
        solver.run(run_params=RUN_PARAMS)
    else:
        solver.run()


def value_error_inf(solver_value, official_value: np.ndarray, center: bool) -> str:
    if solver_value is None:
        return "unavailable"

    value = np.asarray(solver_value, dtype=float)
    if value.ndim == 2:
        value = value.max(axis=1)
    value = value.reshape(-1)

    official = np.asarray(official_value, dtype=float).reshape(-1)
    if value.shape != official.shape:
        raise ValueError(f"value shape {value.shape} != official shape {official.shape}")

    if center:
        value = value - value.mean()
        official = official - official.mean()

    return f"{np.linalg.norm(value - official, ord=np.inf):.6g}"


def solver_files_for(
    solver_filter: Callable[[Path], bool],
    selected_solver: str | None,
) -> list[Path]:
    return sorted(
        path
        for path in SOLVER_PATH.glob("*.py")
        if path.name != "__init__.py"
        and " copy" not in path.stem
        and solver_filter(path)
        and (selected_solver is None or path.stem == selected_solver)
    )


def run_solver_suite(
    model_class,
    solver_filter: Callable[[Path], bool],
    criterion: str,
    default_discount: float,
    center_value_error: bool = False,
):
    parser = argparse.ArgumentParser()
    parser.add_argument("--state", type=int, default=100)
    parser.add_argument("--action", type=int, default=4)
    parser.add_argument("--discount", type=float, default=default_discount)
    parser.add_argument("--precision", type=float, default=1e-2)
    parser.add_argument("--solver", type=str, default=None)
    args = parser.parse_args()

    reference_model = model_class(args.state, args.action)
    reference_model.create_model(save=False)
    official_value = get_exact_value(reference_model, args.discount)

    solver_files = solver_files_for(solver_filter, args.solver)

    print("solver,status,runtime,value_error_inf,error", flush=True)
    for solver_file in solver_files:
        start_time = time.time()
        try:
            model = model_class(args.state, args.action)
            model.create_model(save=False)
            module = load_module(solver_file)
            solver = build_solver(module.Solver, model, args.discount, args.precision)
            run_solver(solver)
            runtime = getattr(solver, "runtime", time.time() - start_time)
            error = value_error_inf(
                getattr(solver, "value", None),
                official_value,
                center_value_error,
            )
            print(f"{solver_file.stem},ok,{runtime:.6f},{error},", flush=True)
        except Exception as exc:
            status = "skipped" if is_optional_backend_import_error(exc) else "failed"
            print(
                f"{solver_file.stem},{status},,,{type(exc).__name__}: {exc}",
                flush=True,
            )


## `exps/test_all_solvers_rooms.py`

Discounted Rooms solver sweep.


In [ ]:
from pathlib import Path

from exps.test_all_solvers_common import run_solver_suite
from mdpforge.models.rooms import Model as RoomsModel


def is_discounted_solver(path: Path) -> bool:
    return (
        "total" not in path.name
        and "average" not in path.name
    )


if __name__ == "__main__":
    run_solver_suite(
        RoomsModel,
        is_discounted_solver,
        criterion="discounted",
        default_discount=0.99,
    )


## `exps/test_all_solvers_total_rooms.py`

Total-reward Rooms solver sweep.


In [ ]:
from pathlib import Path

from test_all_solvers_common import run_solver_suite
from mdpforge.models.rooms_total import Model as RoomsTotalModel


def is_total_solver(path: Path) -> bool:
    return "total" in path.name


if __name__ == "__main__":
    run_solver_suite(
        RoomsTotalModel,
        is_total_solver,
        criterion="total",
        default_discount=1.0,
    )


## `exps/test_all_solvers_average_unichain.py`

Average-reward Unichain solver sweep.


In [ ]:
from pathlib import Path

from test_all_solvers_common import run_solver_suite
from mdpforge.models.unichain import Model as UnichainModel


def is_average_solver(path: Path) -> bool:
    return "average" in path.name


if __name__ == "__main__":
    run_solver_suite(
        UnichainModel,
        is_average_solver,
        criterion="average",
        default_discount=1.0,
        center_value_error=True,
    )


## `exps/test_all_models_personal_vi.py`

Model validation and PersonalVI smoke run.


In [ ]:
import argparse
import importlib.util
import sys
import time
from pathlib import Path


ROOT = Path(__file__).resolve().parents[1]

from mdpforge.utils.paths import MODEL_PATH
from mdpforge.core.validation import validate_model


def load_module(path: Path):
    spec = importlib.util.spec_from_file_location(path.stem, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--state", type=int, default=100)
    parser.add_argument("--action", type=int, default=6)
    parser.add_argument("--discount", type=float, default=0.99)
    parser.add_argument("--precision", type=float, default=1e-2)
    args = parser.parse_args()

    from mdpforge.solvers.personal_vi import Solver as PersonalVI

    model_files = sorted(
        path
        for path in MODEL_PATH.glob("*.py")
        if path.name != "__init__.py"
    )

    print(
        "model,status,state_dim,action_dim,build_time,check_time,runtime,error",
        flush=True,
    )
    for model_file in model_files:
        model = None
        try:
            module = load_module(model_file)
            model = module.Model(args.state, args.action)

            start_time = time.time()
            model.create_model(save=False)
            build_time = time.time() - start_time

            start_time = time.time()
            validate_model(model)
            check_time = time.time() - start_time

            start_time = time.time()
            solver = PersonalVI(model, args.discount, args.precision)
            solver.run()
            runtime = getattr(solver, "runtime", time.time() - start_time)
            print(
                f"{model_file.stem},ok,{model.state_dim},{model.action_dim},"
                f"{build_time:.6f},{check_time:.6f},{runtime:.6f},",
                flush=True,
            )
        except Exception as exc:
            state_dim = getattr(model, "state_dim", "")
            action_dim = getattr(model, "action_dim", "")
            print(
                f"{model_file.stem},failed,{state_dim},{action_dim},,,,"
                f"{type(exc).__name__}: {exc}",
                flush=True,
            )


if __name__ == "__main__":
    main()


## `exps/test_total_models.py`

Small total-reward model check.


In [ ]:
from mdpforge.models.taxi_total import Model
import numpy as np

state_dim = 10000
action_dim = 6

model = Model(state_dim, action_dim)
model.create_model()

print(model.name)
print("State dim : {}".format(model.state_dim))
print("Action dim : {}".format(model.action_dim))
print("Transition density : {}".format(np.round(model.get_transition_density(), 8)))
print("Reward density : {}".format(np.round(model.get_reward_density(), 4)))
print()

discount = 1.0
precision = 1e-2

from mdpforge.solvers.mdptoolbox_vi_total import Solver

solver = Solver(model, discount, precision)
solver.run()
print(f"Runtime: {solver.runtime}")

print()

from mdpforge.solvers.mdptoolbox_pim_total import Solver

solver = Solver(model, discount, precision)
solver.run()
print(f"Runtime: {solver.runtime}")

print(solver.value)


## `notebooks/research/research_best_solver.ipynb`

Research notebook for best-solver comparisons.


In [ ]:
import pandas as pd

df = pd.read_csv('best_solver.csv')
df = df.drop(['Model Name'], axis=1)

df.head()


In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score


label_encoder = LabelEncoder()
df["Best Solver"] = label_encoder.fit_transform(df["Best Solver"])

# Separate features and target
X = df.drop(columns=["Best Solver"])  # Features
y = df["Best Solver"]  # Target

# Convert categorical features into numerical using one-hot encoding (if any)
X = pd.get_dummies(X)

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Create the XGBoost classifier
model = xgb.XGBClassifier(
    objective="multi:softmax",
    num_class=len(label_encoder.classes_),
    use_label_encoder=False,
    eval_metric="mlogloss",
    n_estimators=100,
    max_depth=15,
)

# Train the model
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Convert back to original labels
y_pred_labels = label_encoder.inverse_transform(y_pred)
y_test_labels = label_encoder.inverse_transform(y_test)

# Compute accuracy
accuracy = accuracy_score(y_test_labels, y_pred_labels)
print(f"Accuracy: {accuracy:.2f}")
